# Landslide combined class step 03: post-processing and summaries (minimum scenario)

Runs the landslide post-processing and summary workflow against the combined-class minimum-scenario direct-damage outputs from step 02.


In [ ]:
import re
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter


In [ ]:
base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
output_path = base_path / 'dphil_paper_3/results/02_damage_estimates/landslide_damages/results_landslide_minimum_scenario_combined_class'

data_root = base_path / 'dphil_common_cross_cutting/common_incoming_data'
network_metadata_file = data_root / 'networks/network_layers_hazard_intersections_details.csv'

jamaica_crs = 3448
jmd_per_us_dollar = 150.0

print('Output path:', output_path)
print('Network metadata file:', network_metadata_file)


In [ ]:
damage_results_folder = output_path / 'direct_damages'
damage_estimates_directory = output_path / 'damage_estimates'
damage_estimates_directory.mkdir(parents=True, exist_ok=True)

hazard_column_pattern = re.compile(r'^landslide_combined_class_(baseline|deforestation|reafforestation)_rp_(\d+)$')


def resolve_network_asset_file(asset_relative_path):
    relative_asset_path = Path(asset_relative_path)
    asset_file_in_common_incoming_data = data_root / relative_asset_path
    asset_file_in_nested_networks_folder = data_root / 'networks' / relative_asset_path

    if asset_file_in_common_incoming_data.exists():
        return asset_file_in_common_incoming_data
    if asset_file_in_nested_networks_folder.exists():
        return asset_file_in_nested_networks_folder

    raise FileNotFoundError(
        f"Could not find asset file '{relative_asset_path}'. Checked: {asset_file_in_common_incoming_data} ; {asset_file_in_nested_networks_folder}"
    )


def get_landslide_damage_columns(columns):
    return [column_name for column_name in columns if hazard_column_pattern.match(column_name)]


In [ ]:
asset_data_details = pd.read_csv(network_metadata_file)
damage_totals = []
missing_damage_files = []
damage_rows = []

for asset_info in asset_data_details.itertuples(index=False):
    asset_gpkg = asset_info.asset_gpkg
    asset_layer = asset_info.asset_layer
    asset_id_column = asset_info.asset_id_column

    damage_file = damage_results_folder / f'{asset_gpkg}_{asset_layer}' / f'{asset_gpkg}_{asset_layer}_direct_damages_parameter_set_0_combined_class.parquet'

    if not damage_file.exists():
        missing_damage_files.append(str(damage_file))
        continue

    damage_data = pd.read_parquet(damage_file)
    landslide_damage_columns = get_landslide_damage_columns(damage_data.columns)

    if len(landslide_damage_columns) == 0:
        print(f'No landslide damage columns found in {damage_file.name}, skipping')
        continue

    damage_uncertainty_parameter = (
        damage_data['damage_uncertainty_parameter'].iloc[0]
        if ('damage_uncertainty_parameter' in damage_data.columns and not damage_data.empty)
        else np.nan
    )
    cost_uncertainty_parameter = (
        damage_data['cost_uncertainty_parameter'].iloc[0]
        if ('cost_uncertainty_parameter' in damage_data.columns and not damage_data.empty)
        else np.nan
    )

    for column_name in landslide_damage_columns:
        hazard_column_match = hazard_column_pattern.match(column_name)
        direct_damages_jmd = float(damage_data[column_name].sum())
        damage_rows.append({
            'Sector': asset_info.sector,
            'Subsector': asset_info.asset_description,
            'Asset': asset_gpkg,
            'Layer': asset_layer,
            'Scenario': hazard_column_match.group(1),
            'ReturnPeriod': int(hazard_column_match.group(2)),
            'Direct_Damages_JD': direct_damages_jmd,
            'Direct_Damages_USD': direct_damages_jmd / jmd_per_us_dollar,
        })

    # Asset-level damages
    grouped_damage_data = damage_data.groupby([asset_id_column], as_index=False)[landslide_damage_columns].sum()

    resolved_asset_file = resolve_network_asset_file(asset_info.path)
    asset_geometry_data = gpd.read_file(resolved_asset_file, layer=asset_layer)
    asset_geometry_data = asset_geometry_data.to_crs(epsg=jamaica_crs)

    grouped_damage_data = pd.merge(grouped_damage_data, asset_geometry_data[[asset_id_column, 'geometry']], how='left', on=[asset_id_column])
    grouped_damage_data = gpd.GeoDataFrame(grouped_damage_data, geometry='geometry', crs=jamaica_crs)

    output_geopackage = damage_estimates_directory / f'{asset_gpkg}_{asset_layer}_asset_damages_groupedby_combined_class.gpkg'
    grouped_damage_data.to_file(output_geopackage, driver='GPKG')

    # Sector/layer totals
    asset_damage_totals = grouped_damage_data[landslide_damage_columns].sum().to_frame().T
    asset_damage_totals['sector'] = asset_gpkg
    asset_damage_totals['layer'] = asset_layer
    asset_damage_totals['damage_uncertainty_parameter'] = damage_uncertainty_parameter
    asset_damage_totals['cost_uncertainty_parameter'] = cost_uncertainty_parameter
    damage_totals.append(asset_damage_totals)

if damage_totals:
    damage_totals = pd.concat(damage_totals, axis=0, ignore_index=True)
else:
    damage_totals = pd.DataFrame(columns=['sector', 'layer', 'damage_uncertainty_parameter', 'cost_uncertainty_parameter'])

damage_totals.to_csv(damage_estimates_directory / 'asset_damages_groupedby_combined_class.csv', index=False)

print('Grouped outputs created:', len(damage_totals))
if missing_damage_files:
    print('Missing direct-damage files:', len(missing_damage_files))


In [ ]:
if not damage_estimates_directory.exists():
    raise FileNotFoundError(f'Missing folder: {damage_estimates_directory}')

damage_estimate_files = sorted(damage_estimates_directory.glob('*_asset_damages_groupedby_combined_class.gpkg'))
if not damage_estimate_files:
    raise FileNotFoundError(f'No grouped damage GPKGs found in {damage_estimates_directory}')

summary_rows = []
for grouped_damage_file in damage_estimate_files:
    grouped_damage = gpd.read_file(grouped_damage_file)
    asset_name = grouped_damage_file.stem.replace('_asset_damages_groupedby_combined_class', '')
    summary_rows.append({
        'asset_name': asset_name,
        'row_count': len(grouped_damage),
        'column_count': len(grouped_damage.columns),
    })

pd.DataFrame(summary_rows).sort_values('asset_name').reset_index(drop=True)


In [ ]:
# Choose one output to inspect
asset_name_to_preview = 'roads_edges'  # e.g. rail_nodes, roads_edges, buildings_assigned_economic_activity_areas
preview_file = damage_estimates_directory / f'{asset_name_to_preview}_asset_damages_groupedby_combined_class.gpkg'

if not preview_file.exists():
    raise FileNotFoundError(f'Missing file: {preview_file}')

# Deliberate reread: this standalone preview cell must work after a kernel restart.
preview_table = gpd.read_file(preview_file)
print(f'Rows: {len(preview_table)} | Columns: {len(preview_table.columns)}')
preview_table.head(20)


In [ ]:
# Sum damages by sector, subsector, scenario, and return period.
# The direct-damage files were read once in the grouped-output cell above.
asset_level_summary = pd.DataFrame(damage_rows)

if asset_level_summary.empty:
    raise ValueError('No landslide direct-damage summaries were produced. Run step 02 first.')

sector_subsector_summary = (
    asset_level_summary
    .groupby(['Sector', 'Subsector', 'Scenario', 'ReturnPeriod'], as_index=False)[['Direct_Damages_JD', 'Direct_Damages_USD']]
    .sum()
    .sort_values(['Sector', 'Subsector', 'Scenario', 'ReturnPeriod'])
)

sector_summary = (
    sector_subsector_summary
    .groupby(['Sector', 'Scenario', 'ReturnPeriod'], as_index=False)[['Direct_Damages_JD', 'Direct_Damages_USD']]
    .sum()
    .sort_values(['Sector', 'Scenario', 'ReturnPeriod'])
)

# Scenario deltas vs baseline (both JD and USD)
scenario_delta_summary = (
    sector_subsector_summary
    .pivot_table(index=['Sector', 'Subsector', 'ReturnPeriod'], columns='Scenario', values=['Direct_Damages_JD', 'Direct_Damages_USD'], aggfunc='sum', fill_value=0.0)
)

for unit in ['JD', 'USD']:
    for scenario in ['baseline', 'deforestation', 'reafforestation']:
        if (f'Direct_Damages_{unit}', scenario) not in scenario_delta_summary.columns:
            scenario_delta_summary[(f'Direct_Damages_{unit}', scenario)] = 0.0

scenario_delta_summary = scenario_delta_summary.reset_index()

scenario_delta_summary['Deforestation_Change_JD'] = scenario_delta_summary[('Direct_Damages_JD', 'deforestation')] - scenario_delta_summary[('Direct_Damages_JD', 'baseline')]
scenario_delta_summary['Reafforestation_Change_JD'] = scenario_delta_summary[('Direct_Damages_JD', 'reafforestation')] - scenario_delta_summary[('Direct_Damages_JD', 'baseline')]
scenario_delta_summary['Deforestation_Change_USD'] = scenario_delta_summary[('Direct_Damages_USD', 'deforestation')] - scenario_delta_summary[('Direct_Damages_USD', 'baseline')]
scenario_delta_summary['Reafforestation_Change_USD'] = scenario_delta_summary[('Direct_Damages_USD', 'reafforestation')] - scenario_delta_summary[('Direct_Damages_USD', 'baseline')]

# Flatten multi-index columns after pivot
scenario_delta_summary.columns = [
    column_label if isinstance(column_label, str) else '_'.join([str(part) for part in column_label if str(part) != '']).strip('_')
    for column_label in scenario_delta_summary.columns
]

sector_subsector_summary_file = damage_estimates_directory / 'sector_subsector_scenario_return_period_damages_combined_class.csv'
sector_summary_file = damage_estimates_directory / 'sector_scenario_return_period_damages_combined_class.csv'
delta_summary_file = damage_estimates_directory / 'scenario_deltas_vs_baseline_combined_class.csv'

sector_subsector_summary.to_csv(sector_subsector_summary_file, index=False)
sector_summary.to_csv(sector_summary_file, index=False)
scenario_delta_summary.to_csv(delta_summary_file, index=False)

print(f'Saved: {sector_subsector_summary_file}')
print(f'Saved: {sector_summary_file}')
print(f'Saved: {delta_summary_file}')

print()
print('Sector + Subsector summary:')
display(sector_subsector_summary)
print()
print('Sector-only summary:')
display(sector_summary)
print()
print('Scenario deltas vs baseline:')
display(scenario_delta_summary)


In [ ]:
# RP analysis: damages and avoided damages across sectors
def set_usd_axis(ax, values, base_label):
    flat_values = pd.to_numeric(pd.Series(np.ravel(values)), errors='coerce').replace([np.inf, -np.inf], np.nan).dropna()
    max_abs_value = float(flat_values.abs().max()) if not flat_values.empty else 0.0

    if max_abs_value >= 1_000_000_000:
        scale, suffix, unit_label = 1_000_000_000.0, 'bn', 'USD billion'
    elif max_abs_value >= 1_000_000:
        scale, suffix, unit_label = 1_000_000.0, 'm', 'USD million'
    elif max_abs_value >= 1_000:
        scale, suffix, unit_label = 1_000.0, 'k', 'USD thousand'
    else:
        scale, suffix, unit_label = 1.0, '', 'USD'

    ax.set_ylabel(f'{base_label} ({unit_label})')
    ax.yaxis.set_major_formatter(FuncFormatter(lambda value, position: f'${value / scale:,.2f}{suffix}'))

scenario_order = ['baseline', 'deforestation', 'reafforestation']
sector_order = ['buildings', 'transport', 'water', 'energy']

sector_summary_usd = sector_summary.copy()
sector_summary_usd['ReturnPeriod'] = pd.to_numeric(sector_summary_usd['ReturnPeriod'], errors='coerce').astype(int)
sector_summary_usd['Direct_Damages_USD'] = pd.to_numeric(sector_summary_usd['Direct_Damages_USD'], errors='coerce').fillna(0.0)

# Build a wide table for scenario comparison by sector and RP
sector_rp_scenario = (
    sector_summary_usd
    .pivot_table(
        index=['Sector', 'ReturnPeriod'],
        columns='Scenario',
        values='Direct_Damages_USD',
        aggfunc='sum',
        fill_value=0.0,
    )
    .reset_index()
)

for scenario_name in scenario_order:
    if scenario_name not in sector_rp_scenario.columns:
        sector_rp_scenario[scenario_name] = 0.0

sector_rp_scenario['Avoided_Protection_USD'] = sector_rp_scenario['deforestation'] - sector_rp_scenario['baseline']
sector_rp_scenario['Avoided_Reafforestation_USD'] = sector_rp_scenario['baseline'] - sector_rp_scenario['reafforestation']
sector_rp_scenario['Combined_Benefit_USD'] = sector_rp_scenario['deforestation'] - sector_rp_scenario['reafforestation']

sector_rp_scenario = sector_rp_scenario.sort_values(['Sector', 'ReturnPeriod']).reset_index(drop=True)

# Save tables
sector_rp_scenario_file = damage_estimates_directory / 'sector_return_period_scenario_damages_usd_combined_class.csv'
sector_rp_avoided_file = damage_estimates_directory / 'sector_return_period_avoided_damages_usd_combined_class.csv'

sector_rp_scenario.to_csv(sector_rp_scenario_file, index=False)
sector_rp_scenario[[
    'Sector', 'ReturnPeriod',
    'Avoided_Protection_USD', 'Avoided_Reafforestation_USD', 'Combined_Benefit_USD'
]].to_csv(sector_rp_avoided_file, index=False)

print('Saved:', sector_rp_scenario_file)
print('Saved:', sector_rp_avoided_file)

# Plot 1: direct damages vs RP across sectors (three scenarios)
plot_sectors = [sector_name for sector_name in sector_order if sector_name in sector_rp_scenario['Sector'].unique()]
plot_sectors += sorted([sector_name for sector_name in sector_rp_scenario['Sector'].unique() if sector_name not in plot_sectors])

return_periods = sorted(sector_rp_scenario['ReturnPeriod'].unique().tolist())

number_of_plot_sectors = len(plot_sectors)
ncols = 2
nrows = int(np.ceil(number_of_plot_sectors / ncols))

scenario_styles = {
    'baseline': '#4C78A8',
    'deforestation': '#F58518',
    'reafforestation': '#54A24B',
}

fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4.1 * nrows), sharex=True)
axes = np.array(axes).reshape(-1)

for sector_index, sector_name in enumerate(plot_sectors):
    ax = axes[sector_index]
    sector_data = sector_rp_scenario[sector_rp_scenario['Sector'] == sector_name].sort_values('ReturnPeriod')

    for scenario_name in scenario_order:
        ax.plot(
            sector_data['ReturnPeriod'],
            sector_data[scenario_name],
            marker='o',
            linewidth=2,
            markersize=4,
            color=scenario_styles[scenario_name],
            label=scenario_name.capitalize(),
        )

    ax.set_title(sector_name.capitalize())
    ax.set_xticks(return_periods)
    ax.set_xlabel('Return period (years)')
    set_usd_axis(ax, sector_data[scenario_order].to_numpy().ravel(), 'Direct damages')
    ax.grid(alpha=0.25)

for unused_axis_index in range(number_of_plot_sectors, len(axes)):
    axes[unused_axis_index].set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3, frameon=True, bbox_to_anchor=(0.5, 1.02))
fig.suptitle('Landslide Combined-Class Direct Damages by Return Period Across Sectors', y=1.05)
plt.tight_layout()

direct_rp_chart_file = damage_estimates_directory / 'landslide_sector_direct_damages_by_rp_and_scenario_combined_class.png'
fig.savefig(direct_rp_chart_file, dpi=300, bbox_inches='tight')
print('Saved:', direct_rp_chart_file)
plt.show()

# Plot 2: avoided damages vs RP across sectors
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4.1 * nrows), sharex=True)
axes = np.array(axes).reshape(-1)

for sector_index, sector_name in enumerate(plot_sectors):
    ax = axes[sector_index]
    sector_data = sector_rp_scenario[sector_rp_scenario['Sector'] == sector_name].sort_values('ReturnPeriod')

    ax.plot(
        sector_data['ReturnPeriod'],
        sector_data['Avoided_Reafforestation_USD'],
        marker='o',
        linewidth=2,
        markersize=4,
        color='#2E7D32',
        label='Avoided reafforestation (Baseline - Reafforestation)',
    )
    ax.plot(
        sector_data['ReturnPeriod'],
        sector_data['Avoided_Protection_USD'],
        marker='o',
        linewidth=2,
        markersize=4,
        color='#EF6C00',
        label='Avoided protection (Deforestation - Baseline)',
    )

    ax.axhline(0.0, color='#777777', linewidth=1, linestyle='--')
    ax.set_title(sector_name.capitalize())
    ax.set_xticks(return_periods)
    ax.set_xlabel('Return period (years)')
    set_usd_axis(ax, sector_data[['Avoided_Reafforestation_USD', 'Avoided_Protection_USD']].to_numpy().ravel(), 'Avoided damages')
    ax.grid(alpha=0.25)

for unused_axis_index in range(number_of_plot_sectors, len(axes)):
    axes[unused_axis_index].set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=1, frameon=True, bbox_to_anchor=(0.5, 1.06))
fig.suptitle('Landslide Avoided Damages by Return Period Across Sectors', y=1.10)
plt.tight_layout()

avoided_rp_chart_file = damage_estimates_directory / 'landslide_sector_avoided_damages_by_rp_combined_class.png'
fig.savefig(avoided_rp_chart_file, dpi=300, bbox_inches='tight')
print('Saved:', avoided_rp_chart_file)
plt.show()

display(sector_rp_scenario.head(30))

In [ ]:
# RP bar charts: total and sector direct damages
scenario_order = ['baseline', 'deforestation', 'reafforestation']
scenario_colors = {
    'baseline': '#4C78A8',
    'deforestation': '#F58518',
    'reafforestation': '#54A24B',
}

# Total (all sectors) direct damages by return period
rp_total = (
    sector_rp_scenario
    .groupby('ReturnPeriod', as_index=False)[scenario_order]
    .sum()
    .sort_values('ReturnPeriod')
)

bar_x = np.arange(len(rp_total))
bar_w = 0.24

fig, ax = plt.subplots(figsize=(10.5, 6))
for scenario_index, scenario_name in enumerate(scenario_order):
    ax.bar(
        bar_x + (scenario_index - 1) * bar_w,
        rp_total[scenario_name].to_numpy(),
        width=bar_w,
        label=scenario_name.capitalize(),
        color=scenario_colors[scenario_name],
        edgecolor='white',
        linewidth=0.8,
    )

ax.set_xticks(bar_x)
ax.set_xticklabels([str(int(return_period)) for return_period in rp_total['ReturnPeriod']])
ax.set_xlabel('Return period (years)')
set_usd_axis(ax, rp_total[scenario_order].to_numpy().ravel(), 'Total direct damages')
ax.set_title('Total Landslide Combined-Class Direct Damages by Return Period')
ax.grid(axis='y', alpha=0.25)
ax.legend(title='Scenario', frameon=True)
plt.tight_layout()

total_bar_file = damage_estimates_directory / 'landslide_total_direct_damages_by_rp_bar_combined_class.png'
fig.savefig(total_bar_file, dpi=300, bbox_inches='tight')
print('Saved:', total_bar_file)
plt.show()

# Bar chart for each sector (faceted), direct damages by RP and scenario
sector_order = ['buildings', 'transport', 'water', 'energy']
plot_sectors = [sector_name for sector_name in sector_order if sector_name in sector_rp_scenario['Sector'].unique()]
plot_sectors += sorted([sector_name for sector_name in sector_rp_scenario['Sector'].unique() if sector_name not in plot_sectors])

number_of_plot_sectors = len(plot_sectors)
ncols = 2
nrows = int(np.ceil(number_of_plot_sectors / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4.2 * nrows), sharex=True)
axes = np.array(axes).reshape(-1)

for idx, sector_name in enumerate(plot_sectors):
    ax = axes[idx]
    sector_data = sector_rp_scenario[sector_rp_scenario['Sector'] == sector_name].sort_values('ReturnPeriod')
    x = np.arange(len(sector_data))

    for scenario_index, scenario_name in enumerate(scenario_order):
        ax.bar(
            x + (scenario_index - 1) * bar_w,
            sector_data[scenario_name].to_numpy(),
            width=bar_w,
            color=scenario_colors[scenario_name],
            edgecolor='white',
            linewidth=0.7,
            label=scenario_name.capitalize(),
        )

    ax.set_title(sector_name.capitalize())
    ax.set_xticks(x)
    ax.set_xticklabels([str(int(return_period)) for return_period in sector_data['ReturnPeriod']])
    ax.set_xlabel('Return period (years)')
    set_usd_axis(ax, sector_data[scenario_order].to_numpy().ravel(), 'Direct damages')
    ax.grid(axis='y', alpha=0.25)

for unused_axis_index in range(number_of_plot_sectors, len(axes)):
    axes[unused_axis_index].set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3, frameon=True, bbox_to_anchor=(0.5, 1.02))
fig.suptitle('Sector Direct Damages by Return Period', y=1.05)
plt.tight_layout()

sector_bar_file = damage_estimates_directory / 'landslide_sector_direct_damages_by_rp_bar_panels_combined_class.png'
fig.savefig(sector_bar_file, dpi=300, bbox_inches='tight')
print('Saved:', sector_bar_file)
plt.show()

rp_total